# Convert source json files to the compressed binary format

In [1]:
from glob import glob
from json import loads
import numpy as np
import pandas as pd
import zstandard as zstd

from os import makedirs
from os.path import basename
from simple.utils import pmap

In [24]:
folder = '../json'
opt_folder = '../data/opt'
fut_folder = '../data/opt'

In [17]:
TOrderlog = np.dtype([
    ('ts_event', 'M8[ns]'),
    ('ts_recv', 'M8[ns]'),
    ('action', 'S1'),
    ('side', 'S1'),
    ('price', 'f8'),
    ('size', 'i8'),
    ('channel_id', 'i4'),
    ('order_id', 'u8'),
    ('flags', 'u1'),
    ('ts_in_delta', 'i4'),
    ('sequence', 'i4'),
    ('symbol', 'S45'),
    ('rtype', 'i4'),
    ('publisher_id', 'u4'),
    ('instrument_id', 'i4')
])

TOrderlog.itemsize

112

In [18]:
def iter_orderlog(fname):
    with open(fname, 'rt') as f:
        for s in f:
            j = loads(s)
            hd = j['hd']
            price = j.get('price')
            order_id = j.get('order_id')
            yield (
                np.datetime64(hd['ts_event'][:-1], 'ns'),
                np.datetime64(j['ts_recv'][:-1], 'ns'),
                j['action'],
                j.get('side') or '',
                float(price) if price is not None else np.nan,
                j.get('size') or 0,
                j.get('channel_id') or 0,
                int(order_id) if order_id is not None else 0,
                j.get('flags') or 0,
                j.get('ts_in_delta') or 0,
                j.get('sequence') or 0,
                j.get('symbol') or '',
                hd.get('rtype') or 0,
                hd.get('publisher_id') or 0,
                hd.get('instrument_id') or 0,
            )

In [19]:
def convert(fname, out_folder):
    """repack json file to the zstd-compressed binary formart"""

    L = np.fromiter(iter_orderlog(fname), dtype=TOrderlog)
    makedirs(out_folder, exist_ok=True)
    out_fname = f"{out_folder}/{basename(fname).replace('.json', '.zst')}"

    with open(out_fname, 'wb') as f:
        with zstd.ZstdCompressor(level=18, threads=-1).stream_writer(f) as w:
            w.write(L.tobytes())
            
    return len(L)

In [20]:
def read(fname, dtype=TOrderlog):
    with open(fname, 'rb') as f:
        r = zstd.ZstdDecompressor().stream_reader(f)
        buf = r.readall()
        return np.frombuffer(buf, dtype=dtype)

## Parallel folder conversion

In [21]:
fnames = sorted(glob(f'{folder}/XEUR-20260409-HJTR7RCAKT/xeur-eobi-*.mbo.json'))
fnames

['../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260309.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260310.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260311.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260312.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260313.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260316.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260317.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260318.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260319.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260320.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260323.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260324.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260325.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260326.mbo.json',
 '../json/XEUR-20260409-HJTR7RCAKT/xeur-eobi-20260327.mbo.json',
 '../json/XEUR-20260409-H

In [22]:
print(pmap(convert, fnames, opt_folder))

  0%|          | 0/22 [00:00<?, ?it/s]

[1305607, 749817, 780972, 576394, 625371, 451031, 535900, 688888, 1475162, 742003, 781817, 1006619, 818761, 599945, 477378, 475298, 560527, 796538, 577506, 7616, 507846, 634621]


In [25]:
fnames = sorted(glob(f'{opt_folder}/*.zst'))
fnames

['../data/opt/xeur-eobi-20260309.mbo.zst',
 '../data/opt/xeur-eobi-20260310.mbo.zst',
 '../data/opt/xeur-eobi-20260311.mbo.zst',
 '../data/opt/xeur-eobi-20260312.mbo.zst',
 '../data/opt/xeur-eobi-20260313.mbo.zst',
 '../data/opt/xeur-eobi-20260316.mbo.zst',
 '../data/opt/xeur-eobi-20260317.mbo.zst',
 '../data/opt/xeur-eobi-20260318.mbo.zst',
 '../data/opt/xeur-eobi-20260319.mbo.zst',
 '../data/opt/xeur-eobi-20260320.mbo.zst',
 '../data/opt/xeur-eobi-20260323.mbo.zst',
 '../data/opt/xeur-eobi-20260324.mbo.zst',
 '../data/opt/xeur-eobi-20260325.mbo.zst',
 '../data/opt/xeur-eobi-20260326.mbo.zst',
 '../data/opt/xeur-eobi-20260327.mbo.zst',
 '../data/opt/xeur-eobi-20260330.mbo.zst',
 '../data/opt/xeur-eobi-20260331.mbo.zst',
 '../data/opt/xeur-eobi-20260401.mbo.zst',
 '../data/opt/xeur-eobi-20260402.mbo.zst',
 '../data/opt/xeur-eobi-20260406.mbo.zst',
 '../data/opt/xeur-eobi-20260407.mbo.zst',
 '../data/opt/xeur-eobi-20260408.mbo.zst']

In [28]:
Opt = pd.DataFrame(np.concatenate(pmap(read, fnames))).set_index('ts_event')
Opt

  0%|          | 0/22 [00:00<?, ?it/s]

,ts_recv,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol,rtype,publisher_id,instrument_id
ts_event,,,,,,,,,,,,,,
2026-03-09 07:52:41.367824437,2026-03-09 07:52:41.368148840,b'R',b'N',NaN,0,79,0,8,0,0,b'EUCO SI 20260710 PS EU P 1.1650 0',160,101,34513
2026-03-09 07:52:41.367824437,2026-03-09 07:52:41.368148840,b'A',b'B',0.02120,20,79,10996414798222631105,0,2365,52012,b'EUCO SI 20260710 PS EU P 1.1650 0',160,101,34513
2026-03-09 07:52:41.367824437,2026-03-09 07:52:41.368148840,b'A',b'A',0.02163,20,79,1773042761367855297,128,2365,52012,b'EUCO SI 20260710 PS EU P 1.1650 0',160,101,34513
2026-03-09 07:52:41.367824437,2026-03-09 07:52:41.368148840,b'R',b'N',NaN,0,79,0,8,0,0,b'EUCO SI 20260710 PS EU C 1.1650 0',160,101,34512
2026-03-09 07:52:41.367824437,2026-03-09 07:52:41.368148840,b'A',b'B',0.01920,20,79,10996414798222631105,0,2365,52012,b'EUCO SI 20260710 PS EU C 1.1650 0',160,101,34512
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-08 16:31:31.562120667,2026-04-08 16:31:31.562196101,b'C',b'A',0.00235,20,79,1775665507042983358,128,2444,199054,b'EUCO SI 20260612 PS EU C 1.2125 0',160,101,33321
2026-04-08 16:31:31.562120667,2026-04-08 16:31:31.562196544,b'C',b'B',0.00564,20,79,10999037814277244271,0,1972,199055,b'EUCO SI 20260612 PS EU P 1.1525 0',160,101,33274
2026-04-08 16:31:31.562120667,2026-04-08 16:31:31.562196544,b'C',b'A',0.00609,20,79,1775665777422468463,128,1972,199055,b'EUCO SI 20260612 PS EU P 1.1525 0',160,101,33274


In [29]:
fnames = sorted(glob(f'{folder}/XEUR-20260409-HTT6HHLT6R/xeur-eobi-*.mbo.json'))
fnames

['../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260309.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260310.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260311.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260312.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260313.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260316.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260317.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260318.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260319.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260320.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260323.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260324.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260325.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260326.mbo.json',
 '../json/XEUR-20260409-HTT6HHLT6R/xeur-eobi-20260327.mbo.json',
 '../json/XEUR-20260409-H

## Futures (base asset)

In [30]:
print(pmap(convert, fnames, fut_folder))

  0%|          | 0/22 [00:00<?, ?it/s]

[2232542, 1328747, 1022795, 1032880, 862160, 1645242, 555003, 467447, 910536, 635642, 1340785, 1285000, 1107485, 730748, 599680, 527122, 453503, 708364, 723982, 205, 586606, 1113244]


In [31]:
fnames = sorted(glob(f'{fut_folder}/*.zst'))
fnames

['../data/opt/xeur-eobi-20260309.mbo.zst',
 '../data/opt/xeur-eobi-20260310.mbo.zst',
 '../data/opt/xeur-eobi-20260311.mbo.zst',
 '../data/opt/xeur-eobi-20260312.mbo.zst',
 '../data/opt/xeur-eobi-20260313.mbo.zst',
 '../data/opt/xeur-eobi-20260316.mbo.zst',
 '../data/opt/xeur-eobi-20260317.mbo.zst',
 '../data/opt/xeur-eobi-20260318.mbo.zst',
 '../data/opt/xeur-eobi-20260319.mbo.zst',
 '../data/opt/xeur-eobi-20260320.mbo.zst',
 '../data/opt/xeur-eobi-20260323.mbo.zst',
 '../data/opt/xeur-eobi-20260324.mbo.zst',
 '../data/opt/xeur-eobi-20260325.mbo.zst',
 '../data/opt/xeur-eobi-20260326.mbo.zst',
 '../data/opt/xeur-eobi-20260327.mbo.zst',
 '../data/opt/xeur-eobi-20260330.mbo.zst',
 '../data/opt/xeur-eobi-20260331.mbo.zst',
 '../data/opt/xeur-eobi-20260401.mbo.zst',
 '../data/opt/xeur-eobi-20260402.mbo.zst',
 '../data/opt/xeur-eobi-20260406.mbo.zst',
 '../data/opt/xeur-eobi-20260407.mbo.zst',
 '../data/opt/xeur-eobi-20260408.mbo.zst']

In [32]:
Fut = pd.DataFrame(np.concatenate(pmap(read, fnames)))
Fut

  0%|          | 0/22 [00:00<?, ?it/s]

,ts_event,ts_recv,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol,rtype,publisher_id,instrument_id
0,2026-03-09 00:00:00.000000000,2026-03-09 00:03:00.129732099,b'R',b'N',NaN,0,23,0,8,0,0,b'FCEU SI 20260316 PS',160,101,442
1,2026-03-09 00:00:00.000000000,2026-03-09 00:03:00.129732099,b'A',b'B',1.15280,20,23,10996386599372464268,0,2634,61463,b'FCEU SI 20260316 PS',160,101,442
2,2026-03-09 00:00:00.000000000,2026-03-09 00:03:00.129732099,b'A',b'A',1.15320,20,23,1773014562391096283,128,2634,61463,b'FCEU SI 20260316 PS',160,101,442
3,2026-03-09 00:03:01.430619997,2026-03-09 00:03:01.430638368,b'C',b'A',1.15320,20,23,1773014562391096283,128,1201,61595,b'FCEU SI 20260316 PS',160,101,442
4,2026-03-09 00:03:01.473439135,2026-03-09 00:03:01.473465999,b'A',b'A',1.15330,20,23,1773014581473445312,128,959,61600,b'FCEU SI 20260316 PS',160,101,442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19869713,2026-04-08 20:29:59.286116805,2026-04-08 20:29:59.286131985,b'A',b'A',1.16998,30,23,1775680199286122576,128,995,3575271,b'FCEU SI 20260615 PS',160,101,436
19869714,2026-04-08 20:29:59.319695177,2026-04-08 20:29:59.319717737,b'C',b'B',1.16988,30,23,10999052236140898384,0,1226,3575272,b'FCEU SI 20260615 PS',160,101,436
19869715,2026-04-08 20:29:59.319695177,2026-04-08 20:29:59.319717737,b'A',b'B',1.16987,30,23,10999052236174481524,0,1226,3575272,b'FCEU SI 20260615 PS',160,101,436
19869716,2026-04-08 20:29:59.319695177,2026-04-08 20:29:59.319717737,b'C',b'A',1.16998,30,23,1775680199286122576,0,1226,3575272,b'FCEU SI 20260615 PS',160,101,436


In [33]:
Fut.action.value_counts()

b'C'    9933929
b'A'    9933521
b'T'        991
b'F'        867
b'R'        357
b'M'         53
Name: action, dtype: int64

In [37]:
Fut[Fut.order_id == 10996414336265404691]

,ts_event,ts_recv,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol,rtype,publisher_id,instrument_id
466029,2026-03-09 07:44:59.410622271,2026-03-09 07:44:59.410637352,b'A',b'B',1.1561,20,23,10996414336265404691,128,651,1569926,b'FCEU SI 20260316 PS',160,101,442
466063,2026-03-09 07:45:01.731911761,2026-03-09 07:45:01.735130301,b'F',b'B',1.1561,18,23,10996414336265404691,0,1304,1570115,b'FCEU SI 20260316 PS',160,101,442
466064,2026-03-09 07:45:01.731911761,2026-03-09 07:45:01.735130301,b'C',b'B',1.1561,18,23,10996414336265404691,128,1304,1570115,b'FCEU SI 20260316 PS',160,101,442
466065,2026-03-09 07:45:01.738677045,2026-03-09 07:45:01.738691799,b'C',b'B',1.1561,2,23,10996414336265404691,128,840,1570116,b'FCEU SI 20260316 PS',160,101,442


In [36]:
Fut[Fut.action == b'F']

,ts_event,ts_recv,action,side,price,size,channel_id,order_id,flags,ts_in_delta,sequence,symbol,rtype,publisher_id,instrument_id
466063,2026-03-09 07:45:01.731911761,2026-03-09 07:45:01.735130301,b'F',b'B',1.15610,18,23,10996414336265404691,0,1304,1570115,b'FCEU SI 20260316 PS',160,101,442
488537,2026-03-09 07:55:56.902588117,2026-03-09 07:55:56.905832300,b'F',b'A',1.16163,20,23,1773042956313279001,0,1195,1630888,b'FCEU SI 20260615 PS',160,101,445
488540,2026-03-09 07:55:56.902588117,2026-03-09 07:55:56.905837606,b'F',b'B',1.15659,20,23,10996414993180550785,0,1060,1630889,b'FCEU SI 20260316 PS',160,101,442
488740,2026-03-09 07:56:03.690861135,2026-03-09 07:56:03.693958959,b'F',b'A',1.16162,20,23,1773042963277381107,0,1816,1631332,b'FCEU SI 20260615 PS',160,101,445
488744,2026-03-09 07:56:03.690861135,2026-03-09 07:56:03.693960863,b'F',b'A',-0.00505,20,23,1773042956905648948,0,728,1631333,b'FCEU.S.MAR26.JUN26.SPD',160,101,636539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19645715,2026-04-08 14:49:50.637766025,2026-04-08 14:49:50.640801747,b'F',b'B',1.17355,3,23,10999031826326847992,0,1367,2935688,b'FCEU SI 20260615 PS',160,101,436
19647104,2026-04-08 14:50:31.631649243,2026-04-08 14:50:31.634739297,b'F',b'A',1.17370,20,23,1775659783623755773,0,1647,2939346,b'FCEU SI 20260615 PS',160,101,436
19647278,2026-04-08 14:50:32.844111717,2026-04-08 14:50:32.847183761,b'F',b'A',1.17380,9,23,1775659774351118396,0,2175,2939711,b'FCEU SI 20260615 PS',160,101,436
19648577,2026-04-08 14:51:12.842495717,2026-04-08 14:51:12.845600461,b'F',b'B',1.17380,2,23,10999031908866753128,0,1515,2943064,b'FCEU SI 20260615 PS',160,101,436


In [ ]:
Fut['flags'].value_counts()

In [ ]:
Fut.order_id.value_counts()

In [ ]:
Fut.symbol.value_counts()